# Clase 10 — Correlación y regresión lineal

**Curso:** Estadística Aplicada con Python · **Bloque 4:** Modelos y experimentos
**Tiempo estimado:** 120 minutos · **Capítulo del manual:** `manual/10_correlacion_regresion_lineal.md`

## Qué vas a ver en esta clase

- **Covarianza** y **correlación** de Pearson y de Spearman: qué miden y qué no ven.
- Cuatro conjuntos con **la misma r** y formas muy distintas.
- **Correlación no es causalidad**: una variable de confusión simulada.
- La **regresión lineal por mínimos cuadrados (OLS)**: cálculo a mano y con `statsmodels`.
- Lectura completa del *output*: coeficientes, errores estándar, t, p-value, IC, R², R² corregido, F.
- Regresión **múltiple** ("a igualdad de las demás"), una variable **categórica** (dummy) y la **previsión** con sus dos intervalos.
- **Diagnóstico de los residuos** con datos reales de viviendas de California (con una censura visible), y qué hacer cuando falla.

## Requisitos

- Clases 2 y 3 (media, desviación típica, diagramas de dispersión), 6 (error estándar, IC) y 7 (p-value, t-test).
- Librerías: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; ya vienen en Google Colab).

> **Datos:** el notebook carga `datos/california_housing.csv` (5 000 distritos, muestra aleatoria con semilla 42 del dataset *California housing*, Pace y Barry, 1997, en la versión de scikit-learn). Si no lo encuentra, lo descarga con scikit-learn; y si tampoco puede, genera un dataset **sintético** con las mismas columnas (los números cambiarán). Los ejemplos pequeños y las simulaciones son *ejemplos ilustrativos* sintéticos.

---
## Celda 0 — Setup

Importamos las librerías y fijamos la **semilla** del generador aleatorio.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerías cargadas correctamente.")

---
## 1. Covarianza y correlación a mano

Seis personas (*ejemplo ilustrativo*): `horas` de práctica semanal y `nota` en una prueba. Calculamos la covarianza y la correlación de Pearson paso a paso y comprobamos con `numpy`.

In [ ]:
horas = np.array([1, 2, 3, 4, 5, 6], dtype=float)
nota = np.array([2, 3, 5, 4, 7, 7], dtype=float)

dx = horas - horas.mean()
dy = nota - nota.mean()
cov_xy = (dx * dy).sum() / (len(horas) - 1)             # covarianza muestral (ddof=1)
r_mano = (dx * dy).sum() / np.sqrt((dx**2).sum() * (dy**2).sum())

print(f"Suma de productos Sxy = {(dx * dy).sum():.3f}")
print(f"Covarianza = {cov_xy:.3f}")
print(f"r a mano = {r_mano:.4f}")
print(f"r np.corrcoef = {np.corrcoef(horas, nota)[0, 1]:.4f}")
print(f"r scipy pearsonr = {stats.pearsonr(horas, nota)[0]:.4f}")

**Observa:** la covarianza (3,6) depende de las unidades (horas × puntos); r = 0,93 no: siempre está entre −1 y 1. Que sea positiva dice que a más horas, más nota **en estos seis datos**; no que las horas *causen* la nota.

---
## 2. Experimento guiado: la misma r, cuatro historias distintas

Construimos cuatro conjuntos de 20 puntos y **ajustamos cada uno para que r = 0,80** exactamente: una nube lineal con ruido, una parábola, una recta perfecta con un valor atípico y dos grupos sin relación interna. La función `ajustar` busca con `brentq` el parámetro que da r = 0,80.

**Antes de ejecutar, escribe:** ¿crees que r = 0,80 describe igual de bien los cuatro?

In [ ]:
from scipy.optimize import brentq

def r_de(x, y):
    return np.corrcoef(x, y)[0, 1]

xs = np.linspace(0, 10, 20)
ruido = rng.normal(0, 1, 20)
grupo = np.repeat([0, 1], 10)
x_grupos = np.concatenate([rng.normal(3, 0.8, 10), rng.normal(8, 0.8, 10)])
ruido_grupos = rng.normal(0, 1, 20)
x_atip = np.append(np.linspace(0, 9, 19), 10.0)

def forma_lineal(s):   return xs, 1 + 0.8 * xs + s * ruido
def forma_parabola(c): return xs, -(xs - c) ** 2
def forma_atipico(v):  return x_atip, np.append(1 + 0.8 * x_atip[:19] + 0.3 * ruido[:19], v)
def forma_grupos(d):   return x_grupos, ruido_grupos + d * grupo

def ajustar(funcion, lo, hi, r_obj=0.80):
    """Busca el parámetro que hace que la correlación valga r_obj."""
    p = brentq(lambda t: r_de(*funcion(t)) - r_obj, lo, hi)
    return funcion(p)

conjuntos = {
    "Nube lineal con ruido": ajustar(forma_lineal, 0.01, 30),
    "Parábola": ajustar(forma_parabola, 5.0001, 60),
    "Recta + un atípico": ajustar(forma_atipico, -500, 9),
    "Dos grupos": ajustar(forma_grupos, 0.01, 50),
}

fig, ejes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, (nombre, (x, y)) in zip(ejes, conjuntos.items()):
    b1, b0 = np.polyfit(x, y, 1)
    ax.scatter(x, y, color="steelblue", s=25)
    ax.plot(xs, b0 + b1 * xs, color="crimson")
    ax.set_title(f"{nombre}\nr = {r_de(x, y):.2f}", fontsize=9)
plt.tight_layout(); plt.show()

for nombre, (x, y) in conjuntos.items():
    print(f"{nombre:<32} Pearson = {r_de(x, y):.2f}   Spearman = {stats.spearmanr(x, y)[0]:.2f}")

### ¿Qué observas?

- Las cuatro r valen 0,80, pero solo la primera es "una nube lineal". La **parábola** no es una recta: la relación es perfecta y curva. En la cuarta, la correlación viene **solo de la separación entre grupos**; dentro de cada grupo no hay relación.
- Un solo punto (el atípico) rebaja r de casi 1 a 0,80 (tercer gráfico). Spearman, que usa **rangos**, es menos sensible a él.
- Conclusión: **r resume; no sustituye al gráfico.** Dibuja siempre la nube antes de fiarte del número.

---
## 3. Pearson frente a Spearman: relación monótona pero no lineal

Spearman es la correlación de Pearson calculada sobre los **rangos**. Mide relaciones **monótonas** (siempre crecientes o decrecientes), aunque no sean rectas.

In [ ]:
x_m = np.arange(1, 11, dtype=float)
y_m = np.exp(x_m / 2)                     # crecimiento exponencial: monótono, no lineal

rangos_x, rangos_y = stats.rankdata(x_m), stats.rankdata(y_m)
print(f"Pearson  = {r_de(x_m, y_m):.3f}")
print(f"Spearman = {stats.spearmanr(x_m, y_m)[0]:.3f}")
print(f"Pearson sobre los rangos = {r_de(rangos_x, rangos_y):.3f}")

**Observa:** Spearman vale exactamente 1 (el orden se conserva) y Pearson queda por debajo porque la relación no es una recta. Si Spearman es alto y Pearson bajo, sospecha una relación curva o un atípico.

---
## 4. Correlación no es causalidad: una variable de confusión

Simulamos (*ejemplo ilustrativo*) una **temperatura** `z` que aumenta tanto las `ventas` de helados `x` como los `incidentes` de playa `y`. **No hay ningún efecto de `x` sobre `y`** en la simulación: lo sabemos porque lo hemos construido así.

**Antes de ejecutar:** ¿crees que `x` e `y` estarán correlacionadas? ¿Qué pasará al "controlar" por `z`?

In [ ]:
g = np.random.default_rng(7)                     # generador propio para esta simulación
n_sim = 2000
z = g.normal(0, 1, n_sim)                      # temperatura (estandarizada)
x = z + g.normal(0, 1, n_sim)                  # ventas de helados: depende de z
y = z + g.normal(0, 1, n_sim)                  # incidentes en la playa: depende de z, NO de x
sim = pd.DataFrame({"x": x, "y": y, "z": z})

print(f"Correlación cruda de x con y: {r_de(x, y):.3f}")

m_crudo = smf.ols("y ~ x", data=sim).fit()
m_ajustado = smf.ols("y ~ x + z", data=sim).fit()
for nombre, m in [("y ~ x", m_crudo), ("y ~ x + z", m_ajustado)]:
    print(f"{nombre:<10} coef. de x = {m.params['x']:+.3f}   error estándar = {m.bse['x']:.3f}   p = {m.pvalues['x']:.2g}")

# Estratificar: correlación de x con y dentro de 10 tramos de temperatura
sim["tramo"] = pd.qcut(sim["z"], 10, labels=False)
r_tramos = [r_de(t["x"], t["y"]) for _, t in sim.groupby("tramo")]
print(f"Correlación media de x con y dentro de cada tramo de temperatura: {np.mean(r_tramos):.3f}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].scatter(sim["x"], sim["y"], s=6, alpha=0.4)
ejes[0].set_title("Cruda: x e y parecen relacionadas"); ejes[0].set_xlabel("x"); ejes[0].set_ylabel("y")
sc = ejes[1].scatter(sim["x"], sim["y"], c=sim["z"], s=6, cmap="coolwarm")
ejes[1].set_title("Coloreada por temperatura z"); ejes[1].set_xlabel("x")
plt.colorbar(sc, ax=ejes[1]); plt.tight_layout(); plt.show()

### ¿Qué observas?

- La correlación cruda es claramente positiva (≈ 0,5) y el coeficiente de `x` es "muy significativo"... **sin ningún efecto causal**.
- Al añadir `z` al modelo, el coeficiente de `x` cae a casi 0 y su p-value ya no es pequeño. Dentro de cada tramo estrecho de temperatura, la correlación media de `x` con `y` es cercana a 0 (mucho menor que la cruda).
- Un análisis observacional solo puede "controlar" las variables que **mide**. Si `z` no estuviera en los datos, no veríamos el problema. La causalidad se establece con diseño (clase 13), no con un coeficiente.

---
## 5. Mínimos cuadrados a mano

La recta `nota ≈ b0 + b1 · horas` que minimiza la **suma de los cuadrados de los residuos** tiene pendiente `b1 = Sxy / Sxx` e intercepto `b0 = media(y) − b1 · media(x)`. Con los seis datos de antes:

In [ ]:
Sxx = (dx**2).sum()
Sxy = (dx * dy).sum()
b1 = Sxy / Sxx
b0 = nota.mean() - b1 * horas.mean()
pred = b0 + b1 * horas
resid = nota - pred

print(f"Sxx = {Sxx:.2f}   Sxy = {Sxy:.2f}")
print(f"b1 (pendiente) = {b1:.4f}   b0 (intercepto) = {b0:.4f}")
print("Residuos:", np.round(resid, 3))
print(f"Suma de los residuos (en valor absoluto) = {abs(resid.sum()):.3f}")
print(f"Suma de cuadrados de los residuos = {(resid**2).sum():.4f}")
print("np.polyfit:", np.round(np.polyfit(horas, nota, 1), 4))

**Experimento:** ¿de verdad la pendiente `b1` es la que minimiza la suma de cuadrados? Probamos otras pendientes (recolocando el intercepto para que la recta pase por el centro de los datos) y comparamos.

In [ ]:
pendientes = np.round(np.arange(0.6, 1.5, 0.1), 2)
filas = []
for p in pendientes:
    interc = nota.mean() - p * horas.mean()
    filas.append((p, ((nota - (interc + p * horas)) ** 2).sum()))
tabla_pend = pd.DataFrame(filas, columns=["pendiente", "SSR"])
print(tabla_pend.round(3).to_string(index=False))

grid = np.linspace(0.4, 1.7, 100)
ssr = [((nota - ((nota.mean() - p * horas.mean()) + p * horas)) ** 2).sum() for p in grid]
plt.plot(grid, ssr, color="steelblue")
plt.axvline(b1, color="crimson", linestyle="--", label=f"b1 = {b1:.3f}")
plt.xlabel("Pendiente probada"); plt.ylabel("Suma de cuadrados de los residuos")
plt.title("La pendiente OLS es el mínimo de la parábola"); plt.legend(); plt.show()

**¿Qué observas?** La suma de cuadrados forma una parábola cuyo mínimo cae exactamente en `b1 = Sxy/Sxx`. "Mínimos cuadrados" significa esto: entre todas las rectas, la elegida deja la menor suma de errores al cuadrado. Con dos variables, además, R² = r² (0,93² ≈ 0,87).

---
## 6. Datos reales: viviendas de California

Cada fila es un **distrito** del censo de 1990. `MedHouseVal` es el valor mediano de las viviendas en **unidades de 100 000 \$**; `MedInc`, la renta mediana en decenas de miles de \$; `HouseAge`, la edad mediana de las viviendas; `AveRooms` y `AveOccup`, habitaciones y ocupantes medios por vivienda. El objetivo está **censurado**: los valores altos se recortan en 5,00001. Lo comprobamos.

In [ ]:
def generar_california_sintetico(n=5000, seed=42):
    """Genera un dataset SINTÉTICO con las mismas columnas que California housing (alternativa si no hay CSV ni descarga).
    Las relaciones están inventadas: sirve para practicar, no para sacar conclusiones sobre California."""
    g = np.random.default_rng(seed)
    MedInc = np.clip(g.lognormal(1.25, 0.5, n), 0.5, 15)
    HouseAge = g.integers(1, 53, n).astype(float)
    AveRooms = np.clip(3.0 + 0.35 * MedInc + g.normal(0, 0.7, n), 2, 9)
    AveBedrms = np.clip(0.2 * AveRooms + g.normal(0, 0.05, n), 0.6, 2)
    Population = np.round(g.lognormal(7.0, 0.6, n))
    AveOccup = np.clip(g.lognormal(1.05, 0.25, n), 1.2, 5.5)
    Latitude = np.round(g.uniform(32.5, 42.0, n), 2)
    Longitude = np.round(g.uniform(-124.3, -114.5, n), 2)
    valor = (0.5 + 0.42 * MedInc + 0.012 * HouseAge - 0.15 * AveRooms - 0.3 * AveOccup
             + 0.05 * (37 - Latitude) + g.normal(0, 0.6, n))
    MedHouseVal = np.clip(valor, 0.15, 5.00001)          # censura en 5,00001, como el original
    return pd.DataFrame({"MedInc": MedInc, "HouseAge": HouseAge, "AveRooms": AveRooms, "AveBedrms": AveBedrms,
                         "Population": Population, "AveOccup": AveOccup, "Latitude": Latitude,
                         "Longitude": Longitude, "MedHouseVal": MedHouseVal})


def cargar_california():
    """Devuelve (DataFrame, origen): CSV local, si no descarga de scikit-learn, si no datos sintéticos."""
    for ruta in ("../datos/california_housing.csv", "datos/california_housing.csv", "california_housing.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "CSV local"
    try:
        from sklearn.datasets import fetch_california_housing
        datos = fetch_california_housing(as_frame=True).frame
        return datos.sample(n=5000, random_state=42).reset_index(drop=True), "scikit-learn"
    except Exception:
        return generar_california_sintetico(), "sintético"


df_bruto, origen = cargar_california()
print("Origen de los datos:", origen, "|", df_bruto.shape[0], "filas")

In [ ]:
# Preparación: quitamos distritos con valores medios absurdos y creamos la variable de zona
df = df_bruto[(df_bruto["AveRooms"] < 10) & (df_bruto["AveOccup"] < 6)].copy()
df["zona"] = np.where(df["Latitude"] >= 36, "Norte", "Sur")
print(f"Distritos usados: {len(df)} de {len(df_bruto)}")

In [ ]:
print(df[["MedHouseVal", "MedInc", "HouseAge", "AveRooms", "AveOccup"]].describe().round(2).to_string())

techo = df["MedHouseVal"] >= 5
print(f"\nDistritos en el tope (MedHouseVal ≥ 5): {techo.sum()} ({techo.mean():.1%})")
print(f"Valor máximo: {df['MedHouseVal'].max()}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
ejes[0].hist(df["MedHouseVal"], bins=50, color="steelblue")
ejes[0].set_title("MedHouseVal: la barra final es la censura"); ejes[0].set_xlabel("Valor (100 000 $)")
ejes[1].scatter(df["MedInc"], df["MedHouseVal"], s=4, alpha=0.3)
ejes[1].set_title("Valor frente a renta mediana"); ejes[1].set_xlabel("MedInc"); ejes[1].set_ylabel("MedHouseVal")
plt.tight_layout(); plt.show()

print(f"\nCorrelación de Pearson MedInc–MedHouseVal: {r_de(df['MedInc'], df['MedHouseVal']):.3f}")
print(f"Correlación de Spearman MedInc–MedHouseVal: {stats.spearmanr(df['MedInc'], df['MedHouseVal'])[0]:.3f}")

**Observa:** hay una pila de distritos exactamente en el tope: son valores **recortados**, no medidos. En el diagrama de dispersión se ven como una franja horizontal arriba. Volveremos a ella en los residuos.

---
## 7. Regresión simple con `statsmodels` y lectura del output

`smf.ols("MedHouseVal ~ MedInc", data=df)` ajusta la recta; `.fit()` calcula. Para que el resultado sea reproducible (sin fecha ni hora) imprimimos la tabla de coeficientes y las medidas globales por separado.

In [ ]:
def resumen(m):
    """Imprime la tabla de coeficientes y las medidas globales de un modelo OLS."""
    print(m.summary().tables[1])
    print(f"n = {int(m.nobs)}   R² = {m.rsquared:.4f}   R² corregido = {m.rsquared_adj:.4f}   F = {m.fvalue:.1f}   p(F) = {m.f_pvalue:.2g}")

m1 = smf.ols("MedHouseVal ~ MedInc", data=df).fit()
resumen(m1)

In [ ]:
# Reconstruimos a mano lo que muestra la tabla
b0_m1, b1_m1 = m1.params["Intercept"], m1.params["MedInc"]
ee_m1 = m1.bse["MedInc"]
t_m1 = b1_m1 / ee_m1
gl = int(m1.df_resid)
t_crit = stats.t.ppf(0.975, gl)
print(f"t = coef / ee = {b1_m1:.4f} / {ee_m1:.4f} = {t_m1:.2f}")
print(f"IC 95 % a mano = ({b1_m1 - t_crit * ee_m1:.4f}, {b1_m1 + t_crit * ee_m1:.4f})   grados de libertad = {gl}")
print(m1.conf_int().round(4).to_string())
print(f"r² de Pearson = {r_de(df['MedInc'], df['MedHouseVal'])**2:.4f}   R² del modelo = {m1.rsquared:.4f}")
print(f"Efecto de +1 unidad de MedInc (10 000 $): {b1_m1:+.3f} × 100 000 $ = {b1_m1 * 100_000:.0f} $")

### ¿Qué observas?

- La pendiente es el cambio medio de `MedHouseVal` por cada unidad de `MedInc`; el intercepto es el valor **predicho** con `MedInc = 0`, que está fuera de la zona donde hay datos: no lo interpretes literalmente.
- t = coef/ee y el IC = coef ± t·ee coinciden con la tabla. Con miles de datos el p-value sale "0,000": significa "menor que 0,0005", no "cero".
- Con una sola variable, **R² = r²**. R² es la fracción de la variabilidad de `y` que la recta explica, **no** "la probabilidad de acertar".

---
## 8. Regresión múltiple, variable categórica y R² corregido

Añadimos `HouseAge`, `AveRooms`, `AveOccup` y la zona (`C(zona)`, con Norte como categoría de referencia si `Latitude ≥ 36`, y Sur en caso contrario). Cada coeficiente se lee **a igualdad de las demás variables**.

In [ ]:
m2 = smf.ols("MedHouseVal ~ MedInc + HouseAge + AveRooms + AveOccup + C(zona)", data=df).fit()
resumen(m2)

print(f"\nCorrelación simple AveRooms–MedHouseVal: {r_de(df['AveRooms'], df['MedHouseVal']):+.3f}")
print(f"Correlación MedInc–AveRooms: {r_de(df['MedInc'], df['AveRooms']):+.3f}")
print(f"Coeficiente de AveRooms en el modelo múltiple: {m2.params['AveRooms']:+.3f}")

### ¿Qué observas?

- `AveRooms` tiene correlación **positiva** con el valor, pero su coeficiente en el modelo múltiple es **negativo**. No es un error: a igual renta, ocupación y edad, más habitaciones medias se asocian con menor valor. `AveRooms` y `MedInc` están correlacionadas, y el modelo simple mezclaba ambos efectos.
- El coeficiente de la zona compara **Sur con Norte** (la categoría de referencia), a igualdad del resto.
- Interpretar un coeficiente sin el "a igualdad de las demás" es un error frecuente.

### Significativo no es lo mismo que importante

Un p-value pequeño dice que un coeficiente es **distinguible de cero**, no que su efecto sea **grande**. Comparamos, en un modelo con `Population`, el efecto de un cambio de **una desviación típica** en `MedInc` y en `Population`.

In [ ]:
m_pob = smf.ols("MedHouseVal ~ MedInc + HouseAge + AveRooms + AveOccup + Population", data=df).fit()
for v in ("MedInc", "Population"):
    efecto_sd = m_pob.params[v] * df[v].std()
    print(f"{v:<11} coef = {m_pob.params[v]:.6f}   p = {m_pob.pvalues[v]:.2g}   efecto de +1 desv. típ.: {efecto_sd:+.3f} (×100 000 $)")

**Observa:** los dos coeficientes tienen un p-value diminuto, pero un cambio típico de `Population` mueve el valor una fracción de lo que mueve un cambio típico de `MedInc`. Con miles de datos casi todo sale "significativo": pregunta siempre **cuánto** cambia el resultado, no solo si el p-value es menor que 0,05.

### Experimento guiado: R² siempre sube, R² corregido no

Tomamos **100 distritos** y añadimos columnas de **ruido puro** (números aleatorios sin relación con nada). ¿Qué pasa con R² y con R² corregido? **Antes de ejecutar**, escribe tu predicción.

In [ ]:
pequena = df.sample(100, random_state=1).reset_index(drop=True)
X_base = pequena[["MedInc", "HouseAge", "AveOccup"]]
filas = []
for k in (0, 5, 10, 20, 40):
    r2, r2aj = [], []
    for _ in range(200):                               # 200 repeticiones con ruido distinto
        X_k = pd.concat([X_base, pd.DataFrame(rng.normal(0, 1, (100, k)), columns=[f"r{i}" for i in range(k)])], axis=1)
        m = sm.OLS(pequena["MedHouseVal"], sm.add_constant(X_k)).fit()
        r2.append(m.rsquared); r2aj.append(m.rsquared_adj)
    filas.append((k, np.mean(r2), np.mean(r2aj)))
print(pd.DataFrame(filas, columns=["variables de ruido", "R² medio", "R² corregido medio"]).round(3).to_string(index=False))

### ¿Qué observas?

R² **sube siempre** al añadir variables, aunque sean puro ruido: con 40 columnas inútiles y solo 100 filas parece que el modelo "explica" mucho más. El R² corregido penaliza cada variable extra y no premia el ruido. Por eso **R² alto no prueba que el modelo sea correcto**, y comparar modelos con distinto número de variables exige el R² corregido (o criterios como el AIC, clase 12).

---
## 9. Previsión: dos intervalos distintos

Para una renta `MedInc` dada hay dos preguntas distintas:

- ¿Cuál es el **valor medio** de los distritos con esa renta? → **IC de la media** (estrecho).
- ¿Dónde caerá el valor de **un distrito concreto** con esa renta? → **intervalo de predicción** (más ancho: añade la dispersión individual alrededor de la recta).

Usamos el modelo simple `m1`. Incluimos un valor **fuera del rango** de los datos para ver por qué extrapolar es peligroso.

In [ ]:
nuevos = pd.DataFrame({"MedInc": [2.0, 4.0, 6.0, 10.0, 25.0]})
pf = m1.get_prediction(nuevos).summary_frame(alpha=0.05)
pf.insert(0, "MedInc", nuevos["MedInc"])
pf["ancho IC media"] = pf["mean_ci_upper"] - pf["mean_ci_lower"]
pf["ancho int. predicción"] = pf["obs_ci_upper"] - pf["obs_ci_lower"]
print(pf[["MedInc", "mean", "mean_ci_lower", "mean_ci_upper", "obs_ci_lower", "obs_ci_upper"]].round(2).to_string(index=False))
print(f"\nRango observado de MedInc: {df['MedInc'].min():.2f} – {df['MedInc'].max():.2f}")

rejilla = pd.DataFrame({"MedInc": np.linspace(0.5, 12, 100)})
pg = m1.get_prediction(rejilla).summary_frame(alpha=0.05)
plt.scatter(df["MedInc"], df["MedHouseVal"], s=3, alpha=0.2, color="gray")
plt.plot(rejilla["MedInc"], pg["mean"], color="crimson", label="recta ajustada")
plt.fill_between(rejilla["MedInc"], pg["mean_ci_lower"], pg["mean_ci_upper"], color="crimson", alpha=0.4, label="IC 95 % de la media")
plt.plot(rejilla["MedInc"], pg["obs_ci_lower"], color="black", linestyle="--", linewidth=1, label="intervalo de predicción 95 %")
plt.plot(rejilla["MedInc"], pg["obs_ci_upper"], color="black", linestyle="--", linewidth=1)
plt.xlabel("MedInc"); plt.ylabel("MedHouseVal"); plt.legend(); plt.show()

### ¿Qué observas?

- El IC de la media es muy estrecho (con miles de datos conocemos bien la recta); el intervalo de predicción es **mucho más ancho**: un distrito individual varía mucho alrededor de la recta.
- Con `MedInc = 25`, fuera del rango observado, la recta predice un valor imposible (muy por encima del tope de 5). La relación puede no ser lineal más allá de los datos: **no extrapoles**.
- Con `MedInc = 2` el límite inferior del intervalo de predicción es negativo: un valor de vivienda negativo es imposible, otra señal de que el modelo lineal es solo una aproximación.

---
## 10. Diagnóstico de los residuos

Un **residuo** es `y − ŷ`. Si el modelo es adecuado, los residuos parecen ruido sin estructura. Miramos cuatro gráficos sobre el modelo múltiple `m2`:

1. **Residuos frente a valores predichos:** debe verse una nube sin forma (linealidad) y de ancho constante (homocedasticidad).
2. **Gráfico QQ:** compara los residuos con una normal.
3. **Escala-localización:** raíz del residuo estandarizado en valor absoluto frente a lo predicho (otra mirada a la varianza).
4. **Leverage frente a distancia de Cook:** puntos con mucha influencia sobre el ajuste.

Marcamos en rojo los distritos **censurados** (`MedHouseVal ≥ 5`).

In [ ]:
ajustados = m2.fittedvalues
residuos = m2.resid
infl = m2.get_influence()
estand = infl.resid_studentized_internal
palanca = infl.hat_matrix_diag
cook = infl.cooks_distance[0]
col = np.where(df["MedHouseVal"] >= 5, "crimson", "steelblue")

fig, ejes = plt.subplots(2, 2, figsize=(12, 8))
ejes[0, 0].scatter(ajustados, residuos, c=col, s=4, alpha=0.5)
ejes[0, 0].axhline(0, color="black", linewidth=1)
suav = sm.nonparametric.lowess(residuos, ajustados, frac=0.3)
ejes[0, 0].plot(suav[:, 0], suav[:, 1], color="orange", linewidth=2)
ejes[0, 0].set_title("1. Residuos frente a predichos"); ejes[0, 0].set_xlabel("Valor predicho"); ejes[0, 0].set_ylabel("Residuo")

sm.qqplot(estand, line="45", ax=ejes[0, 1], markersize=2)
ejes[0, 1].set_title("2. Gráfico QQ (residuos estandarizados)")

ejes[1, 0].scatter(ajustados, np.sqrt(np.abs(estand)), c=col, s=4, alpha=0.5)
ejes[1, 0].set_title("3. Escala-localización"); ejes[1, 0].set_xlabel("Valor predicho"); ejes[1, 0].set_ylabel("√|residuo estandarizado|")

ejes[1, 1].scatter(palanca, cook, c=col, s=4, alpha=0.5)
ejes[1, 1].set_title("4. Leverage frente a distancia de Cook"); ejes[1, 1].set_xlabel("leverage"); ejes[1, 1].set_ylabel("Distancia de Cook")
plt.tight_layout(); plt.show()

techo = (df["MedHouseVal"] >= 5).values
print(f"Media de los residuos en valor absoluto (siempre ≈ 0 en OLS con intercepto): {abs(residuos.mean()):.3f}")
print(f"Distritos censurados: {techo.sum()}   residuo medio en ellos: {residuos[techo].mean():+.2f}   en el resto: {residuos[~techo].mean():+.2f}")
print(f"Asimetría de los residuos: {stats.skew(residuos):.2f}")
print(f"Mayor distancia de Cook: {cook.max():.3f} ({'distrito'} {int(np.argmax(cook))})   umbral orientativo 4/n: {4 / len(df):.4f}")
print(f"Distritos con Cook > 4/n: {(cook > 4 / len(df)).sum()}")

### ¿Qué observas?

- **Residuos frente a predichos:** los puntos censurados (rojo) forman una **diagonal descendente** arriba: como el valor real está recortado en 5,00001, el residuo es `5 − predicho`. Es la huella de la censura, un problema **real** de estos datos. Además la dispersión no es constante: la nube es más estrecha a la izquierda y más ancha en el centro-derecha (**heterocedasticidad**).
- **QQ:** la cola derecha se separa de la diagonal: los residuos son asimétricos, no normales.
- **Cook:** la mayoría de los distritos tiene distancia de Cook minúscula. Un umbral orientativo como 4/n señala unos cuantos para **mirar**, no para borrar automáticamente.
- **Importante:** la normalidad de los residuos **no es necesaria para que los coeficientes OLS sean buenos estimadores** (mínimos cuadrados no la usa). Sí importa para que los **p-values y los IC** sean fiables cuando la muestra es **pequeña**; con miles de datos el TLC (clase 5) suaviza el problema, pero la heterocedasticidad y la mala forma funcional siguen afectando a la inferencia.

---
## 11. Qué hacer cuando la diagnóstica señala problemas

Tres respuestas habituales, de la más simple a la más cuidadosa:

1. **Errores estándar robustos (HC3):** no cambian los coeficientes, solo los errores estándar (y por tanto t, p e IC), de forma que valen aunque la varianza no sea constante.
2. **Transformar la respuesta** (por ejemplo `log(MedHouseVal)`): suele estabilizar la varianza y reducir la asimetría; los coeficientes pasan a leerse como cambios **relativos** aproximados.
3. **Cambiar la forma funcional** o tratar el problema de fondo (aquí, la censura: un modelo específico para datos censurados, más allá de esta clase).

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

m2_hc3 = smf.ols("MedHouseVal ~ MedInc + HouseAge + AveRooms + AveOccup + C(zona)", data=df).fit(cov_type="HC3")
comparacion = pd.DataFrame({"coef": m2.params, "ee_OLS": m2.bse, "ee_HC3": m2_hc3.bse})
comparacion["ee_HC3 / ee_OLS"] = comparacion["ee_HC3"] / comparacion["ee_OLS"]
print(comparacion.round(4).to_string())

lm, lm_p, f_bp, f_p = het_breuschpagan(m2.resid, m2.model.exog)
print(f"\nTest de Breusch-Pagan (homocedasticidad): LM = {lm:.1f}, p = {lm_p:.2g}")

m_log = smf.ols("np.log(MedHouseVal) ~ MedInc + HouseAge + AveRooms + AveOccup + C(zona)", data=df).fit()
print(f"\nR² modelo en escala original = {m2.rsquared:.3f}   R² modelo con log = {m_log.rsquared:.3f}   (no son comparables directamente)")
print(f"Asimetría de los residuos original / log: {stats.skew(m2.resid):.2f} / {stats.skew(m_log.resid):.2f}")
print(f"Efecto aproximado de +1 en MedInc con el modelo log: {100 * (np.exp(m_log.params['MedInc']) - 1):+.1f} %")

# Sensibilidad: el mismo modelo sin los distritos censurados
m_sin = smf.ols("MedHouseVal ~ MedInc + HouseAge + AveRooms + AveOccup + C(zona)", data=df[df["MedHouseVal"] < 5]).fit()
print(f"\nCoeficiente de MedInc con todos / sin censurados: {m2.params['MedInc']:.3f} / {m_sin.params['MedInc']:.3f}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].scatter(m2.fittedvalues, m2.resid, s=3, alpha=0.3); ejes[0].axhline(0, color="black", linewidth=1)
ejes[0].set_title("Modelo original"); ejes[0].set_xlabel("Predicho"); ejes[0].set_ylabel("Residuo")
ejes[1].scatter(m_log.fittedvalues, m_log.resid, s=3, alpha=0.3); ejes[1].axhline(0, color="black", linewidth=1)
ejes[1].set_title("Modelo con log(MedHouseVal)"); ejes[1].set_xlabel("Predicho")
plt.tight_layout(); plt.show()

### ¿Qué observas?

- Con errores robustos los coeficientes son **idénticos**; cambian los errores estándar (la última columna dice cuánto). Si el cociente es cercano a 1, la heterocedasticidad no estaba afectando mucho a la inferencia.
- Breusch-Pagan rechaza la homocedasticidad, como sugería el gráfico. Con 4 900 datos casi cualquier desviación pequeña sale "significativa": úsalo como **aviso**, no como semáforo.
- El logaritmo reduce la asimetría de los residuos, pero **la censura sigue ahí** (el tope existe también en la escala log). Excluir los distritos censurados cambia la pendiente porque **recorta la muestra según el resultado**: tampoco es una solución limpia. El diagnóstico sirve para **saber qué límites tiene tu modelo** y decirlo.

---
## Resumen de la clase

- **r** mide asociación **lineal** y es sensible a atípicos; **Spearman** mide asociación **monótona** con rangos. La misma r puede esconder formas muy distintas: dibuja siempre el gráfico.
- **Correlación ≠ causalidad:** una variable de confusión crea correlación sin efecto; solo se "controla" lo que se mide.
- **OLS** minimiza la suma de cuadrados de los residuos; el output da coeficientes, ee, t, p-value, IC, R², R² corregido y F.
- En regresión múltiple cada coeficiente es **a igualdad de las demás**; R² sube siempre, R² corregido no.
- Hay dos intervalos de previsión (media y predicción); **no extrapoles** fuera del rango de los datos.
- Diagnostica los residuos: linealidad, varianza, normalidad (importa para la **inferencia** con muestras pequeñas, no para la estimación), influyentes. Con problemas: errores robustos (HC3), transformaciones, otro modelo.

**Siguiente:** clase 11 — regresión logística y modelos lineales generalizados (respuesta binaria, odds ratio, VIF, AIC).